# 00 — Bronze: bids

Ingere a exportação bruta de propostas exatamente como ela chega. **Sem cast,
sem limpeza, sem filtro.** Qualquer coisa que pareça errada aqui é preservada
para que a camada Silver decida o que fazer a respeito, e para que a camada
raw permaneça uma cópia fiel da fonte.

Drift de schema é aceito, não rejeitado: a fonte é uma exportação manual de
Excel cujas colunas mudam sem aviso, e falhar a carga por uma mudança
cosmética pararia o pipeline sem motivo real. O contrato de dados é aplicado
no Silver.

**Nenhuma biblioteca de cluster é necessária.** O arquivo Excel é lido com
pandas + openpyxl (já em `requirements.txt`) e convertido para um Spark
DataFrame, em vez do pacote Maven `com.crealytics.spark.excel` — isso mantém
este notebook executável em qualquer cluster sem nenhuma configuração.

In [0]:
CATALOG = "bronze"
SCHEMA = "bid"
VOLUME_PATH = "/Volumes/raw/bid/bids/bids.xlsx"
TABLE = f"{CATALOG}.{SCHEMA}.bids"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

## Leitura

`dtype=str` é o equivalente em pandas do `inferSchema=false` — tudo chega
como string, e o Silver faz o cast. A leitura é feita pelo nome da aba,
nunca por um intervalo fixo de células: um intervalo fixo como
`Bronze!A1:K1583` trunca silenciosamente no momento em que a fonte cresce
uma linha, que é a pior classe de bug — sem erro, só dado faltando.

In [0]:
%pip install openpyxl

import pandas as pd

pdf_raw = pd.read_excel(VOLUME_PATH, sheet_name="Bronze", dtype=str)
df_raw = spark.createDataFrame(pdf_raw)

print(f"linhas: {df_raw.count()}   colunas: {len(df_raw.columns)}")
df_raw.printSchema()

## Metadados de ingestão

`_ingested_at` e `_source_file` tornam possível responder "quando essa linha
chegou e de onde veio" meses depois, sem o que a triagem de incidentes num
pipeline de dados vira adivinhação.

In [0]:
from pyspark.sql import functions as F

df_bronze = (
    df_raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit(VOLUME_PATH))
)

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")   # deliberado: ver nota abaixo
    .saveAsTable(TABLE)
)

### Sobre o `mergeSchema`

Aceitar evolução de schema no Bronze é uma escolha, não um padrão. A
exportação upstream ganha e perde colunas sem aviso; recusá-las quebraria a
ingestão por uma mudança que não custa nada absorver. O custo é que uma
coluna renomeada chega silenciosamente como uma nova — por isso a checagem
abaixo existe, e por isso o Silver valida contra um contrato explícito.

In [0]:
EXPECTED = {
    "bid_id", "created_at", "created_at_str", "is_confirmed_date", "bid_date",
    "closed_at", "closed_at_str", "outcome", "loss_reason", "competitor_name",
    "client_id", "contract_value_brl",
}

actual = set(df_bronze.columns) - {"_ingested_at", "_source_file"}
missing, unexpected = EXPECTED - actual, actual - EXPECTED

if missing:
    raise ValueError(f"Colunas ausentes na fonte: {sorted(missing)}")
if unexpected:
    print(f"AVISO — novas colunas absorvidas, revisar Silver: {sorted(unexpected)}")

print(f"Checagem de schema aprovada. {spark.table(TABLE).count()} linhas gravadas em {TABLE}.")